# 10 — Gauge $R^{(0)}$: phụ thuộc init hay phụ thuộc checkpoint?

Notebook phân tích **không train** cho weakness W1/Q1 của review: reviewer cho rằng gain của hàng *Fixed $R^{(0)}$* trong bảng gauge mechanism có thể chỉ là "xoay target về hướng thân thiện với init", và yêu cầu control fit $R^{(0)}$ trên một init rồi train các init độc lập khác với cùng $R$.

Trong recipe của paper, student là một checkpoint pretrained, không có projection head, và $R^{(0)}$ được fit ở eval mode trên một calibration subset cố định (`distiller._project_teacher_targets`). Vì vậy $R^{(0)}$ là hàm deterministic của *(checkpoint, corpus)*: seed chỉ đổi data order và dropout. Notebook này đo trực tiếp ba việc trước khi trả tiền GPU cho control training:

1. **Seed**: đọc `gauge_matrix` từ `teacher_projection.pt` của ba seed trong main run, xác nhận chúng trùng nhau và trùng với fit offline.
2. **Checkpoint**: fit $R^{(0)}$ trên các checkpoint 384-d khác (all-MiniLM, paraphrase-MiniLM, MiniLM-L12) và trên chính checkpoint gốc với pooling khác; đo cosine student-target khi dùng $R$ của checkpoint này cho checkpoint kia, khoảng cách giữa các $R$, participation ratio, và phần gain mà rotation rank-one (chỉ khớp hai mean vector) đã giải thích.
3. **Corpus**: fit $R^{(0)}$ trên hai nửa rời nhau của calibration set, đo mức lệch.

Workflow giữ format của `00_main_results.ipynb`: cấu hình tập trung, `PROJECT_DIR` tự phát hiện, teacher cache dùng chung với các run khác, init embeddings cache theo checkpoint để resume, output CSV + dòng LaTeX trong `runs/<RUN_NAME>/`. Kết quả ở đây là tiền đề cho control training `--gauge_fit_student_model` (train MiniLMv2 với $R$ fit trên checkpoint khác) và cho đoạn viết lại Section 5.1.

In [ ]:
# 1. Cấu hình thí nghiệm
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"
AUTO_PULL_REPO, INSTALL_REQUIREMENTS = False, False

# Pair của bảng gauge mechanism: configuration (c). Đổi sang bge_m3_to_minilm_h768
# thì ALT_STUDENTS[768] được dùng thay cho [384].
PAIR = "qwen3_0.6b_to_minilm_h384"
TRAIN_DATA_REL = Path("data/train_set/merged_3_data_5k_each.csv")
MAX_LENGTH = 256
# Cùng giá trị với recipe: 16384 > 14 760 nên calibration set là toàn bộ corpus.
GAUGE_SAMPLES = 16384
ENCODE_BATCH_SIZE = 128
CUDA_VISIBLE_DEVICES = "0"

# Checkpoint thay thế có cùng width. (model_id, pooling). Pooling "cls" là readout
# của recipe; dòng pooling "mean" của chính checkpoint gốc là control "cùng weight,
# khác readout", rẻ nhất trong các control.
ALT_STUDENTS = {
    384: [
        ("sentence-transformers/all-MiniLM-L6-v2", "cls"),
        ("sentence-transformers/paraphrase-MiniLM-L6-v2", "cls"),
        ("microsoft/MiniLM-L12-H384-uncased", "cls"),
    ],
    768: [
        ("google-bert/bert-base-uncased", "cls"),
        ("sentence-transformers/all-mpnet-base-v2", "cls"),
        ("distilbert/distilbert-base-uncased", "cls"),
    ],
}
INCLUDE_MEAN_POOLING_OF_REFERENCE = True

# Thư mục seed của main run (configuration (c), method geoode) để đọc gauge_matrix
# đã lưu. Để trống nếu run nằm trên máy khác; phần 1 sẽ bị bỏ qua có thông báo.
MAIN_SEED_DIRS = [
    # Path("runs/main_results_h200_gpu2_20260903-1650/qwen3_0.6b_to_minilm_h384/geoode/seed_42"),
    # Path("runs/main_results_h200_gpu2_20260903-1650/qwen3_0.6b_to_minilm_h384/geoode/seed_43"),
    # Path("runs/main_results_h200_gpu2_20260903-1650/qwen3_0.6b_to_minilm_h384/geoode/seed_44"),
]

RANDOM_GAUGE_SEEDS = [0, 1, 2]
SAVE_TO_GOOGLE_DRIVE = False
RUN_STAMP = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y%m%d-%H%M%S")
# Điền tên run cũ để dùng lại init-embedding cache; None tạo run mới.
RUN_NAME_OVERRIDE = None
RUN_NAME = RUN_NAME_OVERRIDE or f"analysis_gauge_init_{PAIR}_{RUN_STAMP}"
print(f"Pair: {PAIR}")
print(f"Run: {RUN_NAME}")

In [ ]:
# 2. Repo, dependencies, GPU và dữ liệu
import os
import subprocess
import sys

cwd = Path.cwd().resolve()
PROJECT_DIR = next((p for p in (cwd, cwd.parent) if (p / "main.py").is_file()), None)
if PROJECT_DIR is None:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"

tracked = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "status", "--porcelain", "--untracked-files=no"],
    check=True, capture_output=True, text=True,
).stdout.strip()
if AUTO_PULL_REPO and not tracked:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif AUTO_PULL_REPO:
    print("[git] Bỏ qua pull vì repo có tracked changes.")
if INSTALL_REQUIREMENTS:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
        check=True,
    )
git_head = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
sys.path[:0] = [str(PROJECT_DIR), str(PROJECT_DIR / "notebooks")]
os.environ.setdefault("CUDA_VISIBLE_DEVICES", CUDA_VISIBLE_DEVICES)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from IPython.display import display
from _analysis_common import PAIRS, load_teacher_cache, prewarm_teacher_cache, teacher_cache_path
from src.teacher_projection import (
    fit_gauge_rotation, fit_teacher_projection, project_teacher_embeddings,
    random_orthogonal, retained_energy,
)

try:
    from google.colab import drive as colab_drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
if IN_COLAB and SAVE_TO_GOOGLE_DRIVE:
    colab_drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/embedding-kd-runs")
else:
    OUTPUT_BASE = PROJECT_DIR / "runs"
RUN_ROOT = OUTPUT_BASE / RUN_NAME
CACHE_DIR = OUTPUT_BASE / "teacher_cache"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
assert TRAIN_DATA.is_file(), f"Thiếu training data: {TRAIN_DATA}"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PAIR_CONFIG = PAIRS[PAIR]
REFERENCE_STUDENT = PAIR_CONFIG["student"]
REFERENCE_POOLING = PAIR_CONFIG["student_pooling"]
print(f"Repo: {PROJECT_DIR} @ {git_head}")
print(f"Device: {DEVICE}; output: {RUN_ROOT}")
print(f"Reference student: {REFERENCE_STUDENT} ({REFERENCE_POOLING} pooling)")

In [ ]:
# 3. Teacher cache, PCA target và calibration subset — đúng recipe của distiller
cache_path = prewarm_teacher_cache(
    PROJECT_DIR, pair=PAIR_CONFIG, train_data=TRAIN_DATA, cache_dir=CACHE_DIR,
    max_length=MAX_LENGTH, cuda_visible_devices=CUDA_VISIBLE_DEVICES,
)
teacher_cls, cache_meta = load_teacher_cache(cache_path)
teacher_cls = teacher_cls.float()

df = pd.read_csv(TRAIN_DATA)
text_column = next(c for c in ("premise", "sentence1", "text") if c in df.columns)
texts = df[text_column].astype(str).tolist()
assert len(texts) == teacher_cls.shape[0], (
    f"Cache có {teacher_cls.shape[0]} hàng, corpus có {len(texts)}: cache không khớp corpus"
)

from transformers import AutoConfig
STUDENT_DIM = int(AutoConfig.from_pretrained(REFERENCE_STUDENT).hidden_size)
TEACHER_DIM = int(teacher_cls.shape[1])
# Recipe: PCA fit trên embedding đã centre, nhưng áp dụng không trừ mean, rồi
# renormalize (pca_center_fit=True, pca_subtract_mean=False).
projection, pca_mean = fit_teacher_projection(
    teacher_cls, out_dim=min(STUDENT_DIM, TEACHER_DIM), projection_type="pca", center=True,
)
projection = F.pad(projection, (0, STUDENT_DIM - projection.shape[1])).contiguous()
targets_pca = project_teacher_embeddings(
    teacher_cls, projection, mean=pca_mean, subtract_mean=False, renormalize=True,
)

n_fit = min(len(texts), GAUGE_SAMPLES)
calib_index = torch.linspace(0, len(texts) - 1, n_fit).round().long().unique()
calib_texts = [texts[i] for i in calib_index.tolist()]
T_calib = targets_pca[calib_index]
print(f"Teacher {TEACHER_DIM}-d -> student {STUDENT_DIM}-d, PCA giữ "
      f"{retained_energy(teacher_cls, projection):.1%} energy")
print(f"Calibration subset: {len(calib_index)} / {len(texts)} câu ({text_column})")

In [ ]:
# 4. Init embeddings của từng checkpoint (eval mode, normalize) — cache để resume
from transformers import AutoModel, AutoTokenizer

def slug(model_id, pooling):
    return f"{model_id.replace('/', '__')}__{pooling}"

def encode_init(model_id, pooling, sentences):
    """Bản sao của Distiller._student_initial_embeddings cho một checkpoint bất kỳ."""
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id).to(DEVICE).eval()
    chunks = []
    with torch.no_grad():
        for start in range(0, len(sentences), ENCODE_BATCH_SIZE):
            encoded = tok(
                sentences[start : start + ENCODE_BATCH_SIZE], max_length=MAX_LENGTH,
                truncation=True, padding=True, return_tensors="pt",
            ).to(DEVICE)
            last = model(
                input_ids=encoded["input_ids"], attention_mask=encoded["attention_mask"],
                return_dict=True,
            ).last_hidden_state
            if pooling == "mean":
                mask = encoded["attention_mask"].unsqueeze(-1).to(last.dtype)
                pooled = (last * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            else:
                pooled = last[:, 0, :]
            chunks.append(F.normalize(pooled.float(), dim=-1).cpu())
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return torch.cat(chunks, dim=0)

CHECKPOINTS = [(REFERENCE_STUDENT, REFERENCE_POOLING)]
if INCLUDE_MEAN_POOLING_OF_REFERENCE and REFERENCE_POOLING != "mean":
    CHECKPOINTS.append((REFERENCE_STUDENT, "mean"))
CHECKPOINTS += ALT_STUDENTS[STUDENT_DIM]

INIT = {}
for model_id, pooling in CHECKPOINTS:
    key = slug(model_id, pooling)
    path = RUN_ROOT / f"init_{key}.pt"
    if path.is_file():
        Z = torch.load(path, map_location="cpu")
        print(f"[cache] {key}: {tuple(Z.shape)}")
    else:
        width = int(AutoConfig.from_pretrained(model_id).hidden_size)
        assert width == STUDENT_DIM, f"{model_id} có width {width}, cần {STUDENT_DIM}"
        Z = encode_init(model_id, pooling, calib_texts)
        torch.save(Z, path)
        print(f"[encode] {key}: {tuple(Z.shape)}")
    INIT[key] = Z
REF_KEY = slug(REFERENCE_STUDENT, REFERENCE_POOLING)

## Phần 1 — $R^{(0)}$ giữa ba seed của main run

Nếu ba `gauge_matrix` trùng nhau đến sai số float và trùng với fit offline ở đây, thì hàng *Fixed $R^{(0)}$* của bảng gauge mechanism **đã** là "ba student train độc lập với cùng một $R$ đóng băng", và control cross-seed mà reviewer đề xuất là vacuous. Câu còn lại (Phần 2) là $R$ có đặc thù cho checkpoint hay không.

In [ ]:
# 5. gauge_matrix đã lưu của từng seed so với nhau và so với fit offline
def gauge_from_run(run_dir):
    path = Path(run_dir) / "teacher_projection.pt"
    if not path.is_absolute():
        path = PROJECT_DIR / path
    if not path.is_file():
        return None
    saved = torch.load(path, map_location="cpu", weights_only=False)
    return saved

R_ref_offline, stats_ref = fit_gauge_rotation(T_calib, INIT[REF_KEY], mode="procrustes")

saved_runs = {str(d): gauge_from_run(d) for d in MAIN_SEED_DIRS}
saved_runs = {k: v for k, v in saved_runs.items() if v is not None and v.get("gauge_matrix") is not None}
if not saved_runs:
    print("[skip] Không có teacher_projection.pt nào trong MAIN_SEED_DIRS; "
          "điền đường dẫn seed dirs của main run để chạy phần này.")
    seed_table = pd.DataFrame()
else:
    keys = list(saved_runs)
    rows = []
    for i, a in enumerate(keys):
        Ra = saved_runs[a]["gauge_matrix"].float()
        hist = saved_runs[a].get("gauge_history") or []
        rows.append({
            "run": a,
            "max_abs_diff_vs_seed0": float((Ra - saved_runs[keys[0]]["gauge_matrix"].float()).abs().max()),
            "max_abs_diff_vs_offline": float((Ra - R_ref_offline).abs().max()),
            "cos_after_saved": (saved_runs[a].get("gauge_stats") or {}).get("cos_after"),
            "cos_after_offline": stats_ref["cos_after"],
            "participation_ratio_saved": (saved_runs[a].get("gauge_stats") or {}).get("participation_ratio"),
            "refits_logged": max(0, len(hist) - 1),
        })
    seed_table = pd.DataFrame(rows)
    display(seed_table)
    seed_table.to_csv(RUN_ROOT / "part1_seed_gauge_identity.csv", index=False)

## Phần 2 — $R^{(0)}$ giữa các checkpoint cùng width

Với mỗi checkpoint $j$ ta fit $R_j$ và đo:

- `cos_before`, `cos_after`: cosine student-target trung bình trước và sau khi xoay bằng $R_j$ của chính nó (số mà `main.py` in ra khi train).
- `cos_under_R_ref`: cosine khi dùng $R$ của checkpoint gốc cho checkpoint $j$. Đây là con số mà control training `--gauge_fit_student_model` sẽ thực sự tối ưu, chỉ là đọc ngược: thay vì train MiniLMv2 với $R$ của checkpoint khác, ta xem checkpoint khác dưới $R$ của MiniLMv2.
- `cos_random`: cosine dưới rotation Haar, trung bình trên `RANDOM_GAUGE_SEEDS`; đây là mức của hàng *Random orthogonal*.
- `cos_rank_one`: cosine dưới rotation chỉ khớp hai mean vector. Nếu `cos_rank_one` gần `cos_after` thì gauge chủ yếu sửa hướng anisotropy của init, đúng với `participation_ratio` gần 1.
- `rel_dist_to_R_ref` $= \|R_j - R_{\mathrm{ref}}\|_F / \sqrt{2 d_S}$: 0 là trùng, khoảng 1 là hai rotation không liên quan.
- `init_mean_norm`: $\|\bar z\|$ của init embeddings đã normalize; gần 1 nghĩa là init gần như collapse về một hướng.

In [ ]:
# 6. Fit R trên từng checkpoint, ma trận transfer và các thống kê cơ chế
def mean_cos(T, R, Z):
    return float(((T @ R) * Z).sum(dim=-1).mean())

def rel_dist(Ra, Rb):
    return float((Ra - Rb).norm() / (2 * Ra.shape[0]) ** 0.5)

def init_summary(Z):
    mean = Z.mean(dim=0)
    centred = Z - mean
    ev = torch.linalg.svdvals(centred).pow(2)
    return {
        "init_mean_norm": float(mean.norm()),
        "init_mean_pairwise_cos": float(mean.norm() ** 2),
        "init_effective_dim": float(ev.sum() ** 2 / ev.pow(2).sum().clamp(min=1e-12)),
    }

FITS, rows = {}, []
haar = [random_orthogonal(STUDENT_DIM, seed=s) for s in RANDOM_GAUGE_SEEDS]
for model_id, pooling in CHECKPOINTS:
    key = slug(model_id, pooling)
    Z = INIT[key]
    R, st = fit_gauge_rotation(T_calib, Z, mode="procrustes")
    R1, _ = fit_gauge_rotation(T_calib, Z, mode="rank_one")
    FITS[key] = R
    rows.append({
        "checkpoint": model_id, "pooling": pooling, "key": key,
        "cos_before": st["cos_before"],
        "cos_after": st["cos_after"],
        "cos_under_R_ref": mean_cos(T_calib, R_ref_offline, Z),
        "cos_random": float(np.mean([mean_cos(T_calib, H, Z) for H in haar])),
        "cos_rank_one": mean_cos(T_calib, R1, Z),
        "participation_ratio": st["participation_ratio"],
        "top_singular_share": st["top_singular_share"],
        "rel_dist_to_R_ref": rel_dist(R, R_ref_offline),
        **init_summary(Z),
    })
ckpt_table = pd.DataFrame(rows)
# Phần gain của gauge (so với random) mà rank-one đã giải thích; và phần gain còn
# giữ được khi mượn R của checkpoint gốc.
ckpt_table["rank_one_share_of_gain"] = (
    (ckpt_table["cos_rank_one"] - ckpt_table["cos_random"])
    / (ckpt_table["cos_after"] - ckpt_table["cos_random"]).clip(lower=1e-9)
)
ckpt_table["transfer_share_of_gain"] = (
    (ckpt_table["cos_under_R_ref"] - ckpt_table["cos_random"])
    / (ckpt_table["cos_after"] - ckpt_table["cos_random"]).clip(lower=1e-9)
)
display(ckpt_table.drop(columns=["key"]).round(4))
ckpt_table.to_csv(RUN_ROOT / "part2_checkpoint_gauges.csv", index=False)

# Ma trận transfer đầy đủ: hàng = R fit trên checkpoint i, cột = init của checkpoint j.
keys = [slug(m, p) for m, p in CHECKPOINTS]
transfer = pd.DataFrame(
    [[mean_cos(T_calib, FITS[a], INIT[b]) for b in keys] for a in keys],
    index=[f"R[{k}]" for k in keys], columns=keys,
)
display(transfer.round(4))
transfer.to_csv(RUN_ROOT / "part2_transfer_matrix.csv")

## Phần 3 — $R^{(0)}$ theo corpus: hai nửa rời nhau của calibration set

Reviewer gộp "init" và "corpus" vào cùng một caveat. Fit trên hàng chẵn và hàng lẻ (mỗi nửa ~7 380 câu, vẫn lớn hơn $d_S$ nhiều lần) rồi đo chéo: nếu $R_{\text{even}}$ đạt gần đủ cosine trên nửa lẻ và `rel_dist` nhỏ, gauge không nhạy với subset corpus ở quy mô này. Số này bổ sung cho bảng gauge-fit-set-size sensitivity (2 048 → 14 760) vốn chỉ đo downstream.

In [ ]:
# 7. Ổn định theo corpus split
Z_ref = INIT[REF_KEY]
even = torch.arange(0, len(calib_index), 2)
odd = torch.arange(1, len(calib_index), 2)
R_even, st_even = fit_gauge_rotation(T_calib[even], Z_ref[even], mode="procrustes")
R_odd, st_odd = fit_gauge_rotation(T_calib[odd], Z_ref[odd], mode="procrustes")
split_table = pd.DataFrame([
    {"fit_on": "even", "eval_on": "even", "cos": mean_cos(T_calib[even], R_even, Z_ref[even])},
    {"fit_on": "even", "eval_on": "odd", "cos": mean_cos(T_calib[odd], R_even, Z_ref[odd])},
    {"fit_on": "odd", "eval_on": "odd", "cos": mean_cos(T_calib[odd], R_odd, Z_ref[odd])},
    {"fit_on": "odd", "eval_on": "even", "cos": mean_cos(T_calib[even], R_odd, Z_ref[even])},
    {"fit_on": "full", "eval_on": "full", "cos": stats_ref["cos_after"]},
    {"fit_on": "random", "eval_on": "full", "cos": float(np.mean([mean_cos(T_calib, H, Z_ref) for H in haar]))},
])
split_table["rel_dist_even_odd"] = rel_dist(R_even, R_odd)
split_table["rel_dist_even_full"] = rel_dist(R_even, R_ref_offline)
display(split_table.round(4))
split_table.to_csv(RUN_ROOT / "part3_corpus_split.csv", index=False)

In [ ]:
# 8. Tóm tắt JSON và dòng LaTeX cho bảng appendix "Gauge transfer across checkpoints"
import json

def short_name(model_id):
    return model_id.split("/")[-1]

lines = []
for _, r in ckpt_table.iterrows():
    label = short_name(r["checkpoint"]) + (" (mean)" if r["pooling"] == "mean" else "")
    if r["key"] == REF_KEY:
        label += " [reference]"
    lines.append(
        f"{label} & {r['cos_before']:.3f} & {r['cos_random']:.3f} & {r['cos_rank_one']:.3f} & "
        f"{r['cos_after']:.3f} & {r['cos_under_R_ref']:.3f} & {r['rel_dist_to_R_ref']:.2f} & "
        f"{r['participation_ratio']:.2f} \\\\"
    )
latex = "\n".join([
    "% Columns: checkpoint & cos raw PCA & cos random R & cos rank-one R & cos own R"
    " & cos under reference R & ||R - R_ref||_F / sqrt(2 d_S) & participation ratio",
    "% Source: " + str(RUN_ROOT.relative_to(PROJECT_DIR) if RUN_ROOT.is_relative_to(PROJECT_DIR) else RUN_ROOT),
    *lines,
])
print(latex)
(RUN_ROOT / "gauge_transfer_rows.tex").write_text(latex + "\n", encoding="utf-8")

summary = {
    "pair": PAIR, "git_head": git_head, "run_root": str(RUN_ROOT),
    "calibration_size": int(len(calib_index)),
    "reference": REF_KEY,
    "reference_fit": stats_ref,
    "seed_identity": None if seed_table.empty else {
        "max_abs_diff_between_seeds": float(seed_table["max_abs_diff_vs_seed0"].max()),
        "max_abs_diff_vs_offline": float(seed_table["max_abs_diff_vs_offline"].max()),
    },
    "corpus_split": {
        "rel_dist_even_odd": float(split_table["rel_dist_even_odd"].iloc[0]),
        "cos_cross_split_min": float(split_table.query("fit_on != eval_on and fit_on != 'random'")["cos"].min()),
    },
    "checkpoints": ckpt_table.drop(columns=["key"]).to_dict(orient="records"),
}
(RUN_ROOT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"\nSaved: {RUN_ROOT / 'summary.json'}")

## Cách đọc kết quả và bước tiếp theo

**Phần 1.** `max_abs_diff` giữa các seed ở mức float epsilon là bằng chứng để sửa Section 5.1: hàng *Fixed $R^{(0)}$* đã là ba lần train độc lập với cùng $R$; control cross-seed của reviewer không phân biệt được gì thêm.

**Phần 2.** Hai kịch bản, cả hai đều viết được:

- `transfer_share_of_gain` thấp và `rel_dist_to_R_ref` gần 1 trên các checkpoint khác: $R^{(0)}$ mã hoá hệ toạ độ của *checkpoint cụ thể*. Claim trong bài đổi thành "orients targets into the pretrained student's own frame", và control training với $R$ mượn được dự đoán rơi về mức *Random orthogonal* (73.0).
- `transfer_share_of_gain` cao trong họ MiniLM: gauge chỉ cần một hướng chung của họ checkpoint, claim reusability mạnh hơn.

`rank_one_share_of_gain` và `participation_ratio` nói gauge làm gì: gần 1 nghĩa là $R$ chủ yếu khớp hướng anisotropy của init với hướng PCA trội; đây là cơ chế cụ thể để thay cho câu "cannot distinguish" hiện tại.

**Phần 3.** `rel_dist_even_odd` nhỏ và cosine chéo gần cosine full tách được caveat "corpus" khỏi caveat "init".

**Control training** (chạy sau notebook này): thêm `--gauge_fit_student_model <checkpoint>` để `_student_initial_embeddings` dùng checkpoint khác chỉ ở bước fit, train MiniLMv2 ba seed, thêm một hàng vào bảng gauge mechanism. Kịch bản của hàng đó đã được dự đoán bằng `cos_under_R_ref` ở Phần 2.